![pentest](images/pentest.png "Pentest")

# TP2-ARCH-06 : Redteam, blueteam et chapeaux

## Objectifs pédagogiques

- comprendre les différences dans la sécurité informatique des réseaux
- tester différentes attaques sur le LAN RPi et les moyens de s'en protéger

## Cadre légal et éthique

- ✅ AUTORISÉ dans notre labo :
    - Tester uniquement les machines du réseau isolé
    - Utiliser les comptes de test fournis
    - Documenter toutes les actions dans le cahier de TP

- ❌ INTERDIT (et puni par la loi) :
    - Scanner ou tester des machines hors du labo
    - Utiliser les compétences apprises sur des systèmes réels sans autorisation
    - Partager des outils d'attaque en dehors du cadre pédagogique

- Référence : [Code Pénal Suisse](https://www.fedlex.admin.ch/eli/cc/54/757_781_799/fr)


## Les Concepts expliqués simplement

### Les "Chapeaux" : une question d'éthique

| Chapeau | Couleur | Rôle | Analogie pour comprendre |
|---------|---------|------|-------------------------|
| ⚪ **White Hat** | Blanc | **Gentil hacker** : teste les systèmes *avec autorisation* pour les renforcer | 🔐 Le serrurier qui vérifie votre porte pour vous dire comment mieux la fermer |
| ⚫ **Black Hat** | Noir | **Pirate malveillant** : attaque pour voler, détruire ou nuire *sans autorisation* | 🦹 Le cambrioleur qui force votre serrure pour voler vos affaires |
| 🔘 **Gray Hat** | Gris | **Entre les deux** : teste sans autorisation mais signale les failles (sans malveillance) | 🤷 Quelqu'un qui teste votre porte, trouve la faille, et vous laisse un mot... sans avoir demandé la permission |

> ⚖️ **Rappel légal essentiel** : En Suisse, accéder sans autorisation à un système informatique est un délit pénal (article 143bis du Code pénal). *Toujours avoir une autorisation écrite avant de tester !*

### Red Team vs Blue Team : le jeu du chat et de la souris

- 🔴 RED TEAM = **Les Attaquants** (en entraînement)
  - Objectif : Trouver les failles comme le ferait un vrai pirate
  - Méthodes : Tests d'intrusion, ingénierie sociale, exploitation de vulnérabilités
  - Mentalité : "Comment entrer sans se faire repérer ?"
- 🔵 BLUE TEAM = **Les Défenseurs**
  - Objectif : Détecter, bloquer et réparer les attaques
  - Méthodes : Surveillance, pare-feu, analyse de logs, réponse aux incidents
  - Mentalité : "Comment protéger et réagir vite ?"
- 🟣 PURPLE TEAM = **La collaboration**
  - Red + Blue travaillent ensemble pour améliorer la sécurité globale


> **Analogie sportive** : C'est comme un match d'entraînement au football. La Red Team attaque le but, la Blue Team défend. À la fin, on analyse ensemble pour s'améliorer !


### Le Pentest (Test d'Intrusion) en 5 étapes

1. RECONNAISSANCE → "Qui est ma cible ?"
    - Collecte d'infos publiques (site web, réseaux sociaux...)
1. SCANNING → "Quelles sont les portes d'entrée ?"
    - Recherche des ports ouverts, services actifs, versions logicielles
1. EXPLOITATION → "J'essaie d'entrer !"
    - Utilisation contrôlée d'une faille découverte
1. MAINTIEN D'ACCÈS → "Est-ce que je peux rester ?"
    - Test de persistance (uniquement en environnement autorisé !)
1. RAPPORT → "Voici ce que j'ai trouvé et comment corriger"
    - Document professionnel avec recommandations

## Exercice 1 : "Cartographie du réseau" (Niveau Débutant)
**Objectif** : Comprendre comment on découvre les machines sur un réseau

Il y a un prérequis : installer l'outil de cartographie du réseau :

```bash
sudo apt update
sudo apt install nmap
```

Et commencer la cartographie :

```bash
# Sur le Raspberry "attaquant"
# Commande simple pour scanner le réseau local
nmap -sn 192.168.200.0/24
# Commande simple pour scanner les services ouverts de la machine 192.168.200.XXX
nmap -sV 192.168.200.XXX
```

- Dessiner le schéma du réseau avant/après scan
- Identifier les services ouverts (SSH, HTTP...)
- Rédiger une fiche "profil de cible"

## Exercice 2 : "Le mot de passe fort" (Niveau Intermédiaire)

**Prérequis** : installer john the ripper

```bash
# Mettre à jour les paquets
sudo apt update
# installer snapd (gestionnaire de paquets)
sudo apt install john
# installer la base de données de mots de passes
git clone https://github.com/dw0rsec/rockyou.txt
# extraction
cd rockyou.txt
unzip rockyou.txt.zip
```

Une fois les prérequis techniques installé, nous pouvons passer à l'exercice 2

**Exercice**

```bash
# Création d'un fichier test avec des hashs MD5 simples
# (uniquement sur comptes de test créés pour l'exercice !)
echo "testuser:$(openssl passwd -1 1234)" > /tmp/test_hashes

# Utilisation de john the ripper en mode éducatif
john --wordlist=./rockyou.txt /tmp/test_hashes
```

- Tester 5 mots de passe de complexité variable : dans l'exemple ci-dessus, le mot de passe est `1234`, modifiez-le
- Chronométrer le temps de "craquage"
- Établir une charte "mot de passe robuste" pour l'établissement

## Exercice 3 : Catch the flag (Niveau avancé)

> ⚠️ **AVERTISSEMENT IMPORTANT**  
> Ces commandes sont à exécuter **UNIQUEMENT** dans un réseau isolé (bac à sable) avec des machines dont vous êtes propriétaire ou administrateur (ici, les Raspberry Pi du laboratoire).  
> **Toute utilisation sur des systèmes tiers sans autorisation est illégale selon le code pénal Suisse.**

### 1. Schéma du Réseau de TP

Pour ce TP, nous utilisons au minimum 3 Raspberry Pi connectés au même switch (réseau local isolé). S'il y a plus de groupes, alors on ajoute des attaquants.

| Rôle | Nom de la machine | Adresse IP (Exemple) | Système |
|------|-------------------|----------------------|---------|
| 🔴 **Red Team** (Attaquant) | `Pi-Attaquant` | `192.168.200.10` | Raspberry Pi OS |
| 🎯 **Cible** (Victime) | `Pi-Cible` | `192.168.200.20` | Raspberry Pi OS + Apache |
| 🔵 **Blue Team** (Défenseur) | `Pi-Defense` | `192.168.200.30` | Raspberry Pi OS |


### 2. Préparation de la Machine CIBLE (Pi-Cible)

*Cette machine héberge le "secret" à protéger.*

#### 2.1 : Mise à jour et installation du serveur web
Connectez-vous au `Pi-Cible`.

```bash
# Mise à jour des paquets
sudo apt update

# Installation du serveur web Apache
sudo apt install apache2 -y

# Démarrage et activation du service
sudo systemctl start apache2
sudo systemctl enable apache2
```

#### 2.2 : Création du drapeau (Flag) et de la faille

Nous allons créer un fichier secret dans un dossier peu visible.

```bash
# Création du dossier secret
sudo mkdir -p /var/www/html/confidentiel

# Création du fichier drapeau (le but du jeu)
# Remplacez "FLAG{SECURITE_POUR_TOUS}" par votre propre texte
echo "FLAG{SECURITE_POUR_TOUS}" | sudo tee /var/www/html/confidentiel/flag.txt

# Configuration des droits (Lire pour tous, écrire seulement pour root)
sudo chmod 755 /var/www/html/confidentiel
sudo chmod 644 /var/www/html/confidentiel/flag.txt

# Vérification que le fichier est accessible localement
curl http://localhost/confidentiel/flag.txt
```

#### 2.3 : Préparation pour la Blue Team (Accès aux logs)

La Blue Team devra consulter les logs à distance. Nous créons un utilisateur spécifique pour eux.

```bash
# Création d'un utilisateur 'auditeur'
sudo adduser auditeur
# (Suivez les instructions pour le mot de passe, ex: 'cyber2024')

# Autoriser cet utilisateur à lire les logs Apache sans être root
# On l'ajoute au groupe adm qui a souvent accès aux logs
sudo usermod -aG adm auditeur

# Vérification du chemin des logs
ls -l /var/log/apache2/
```

### 3. Actions de la RED TEAM (Pi-Attaquant)

L'objectif est de trouver et récupérer le fichier `flag.txt` sans connaître son emplacement à l'avance.


#### 3.1 : Reconnaissance (Scanning)

La Red Team commence par découvrir la cible.

```bash
# Installation de l'outil de scan (si pas déjà fait)
sudo apt install nmap -y

# 1. Découverte de la machine cible sur le réseau
# Remplacez 192.168.10.0/24 par votre plage réseau
nmap -sn 192.168.10.0/24

# 2. Scan des ports ouverts sur la cible (192.168.10.20)
nmap -sV 192.168.10.20
```

**Attendu : Le port 80 (http) doit apparaître comme ouvert.**

```bash
# Installation de curl pour interroger le web
sudo apt install curl -y

# 1. Voir la page d'accueil (souvent vide ou par défaut)
curl http://192.168.10.20/

# 2. Tester des noms de dossiers courants (Brute-force manuel)
curl http://192.168.10.20/admin/
curl http://192.168.10.20/secret/
curl http://192.168.10.20/confidentiel/
```
Si le dossier existe, le contenu s'affiche.

#### 3.3 : Exfiltration (Vol du drapeau)

Une fois le fichier trouvé, la Red Team le récupère.

```bash
# Téléchargement du drapeau sur la machine attaquante
curl http://192.168.10.20/confidentiel/flag.txt -o flag_vole.txt

# Vérification du contenu
cat flag_vole.txt
```

**Mission Red Team Accomplie : Vous avez le drapeau. Notez l'heure exacte de l'attaque pour le debrief.**

### 4. Actions de la BLUE TEAM (Pi-Defense)

L'objectif est de détecter l'attaque, identifier l'attaquant et comprendre ce qu'il a fait.

#### 4.1 : Connexion à la cible pour analyse

La Blue Team ne travaille pas sur la cible directement, elle s'y connecte pour analyser.

```bash
# Depuis le Pi-Defense, connexion SSH au Pi-Cible
# Remplacez 'auditeur' et l'IP par les vôtres
ssh auditeur@192.168.10.20
# (Entrez le mot de passe défini à l'étape 2.3)
```

#### 4.2 : Analyse des logs d'accès (Apache)

```bash
# Une fois connecté en SSH sur la Cible :

# 1. Voir les dernières lignes du log d'accès
tail -n 20 /var/log/apache2/access.log

# 2. Chercher spécifiquement l'adresse IP de l'attaquant (192.168.10.10)
grep "192.168.10.10" /var/log/apache2/access.log

# 3. Chercher les tentatives d'accès au fichier flag
grep "flag.txt" /var/log/apache2/access.log
```

#### 4.3 : Rapport d'incident

La Blue Team doit rédiger un rapport basé sur ces commandes.

**Exemple de ce qu'ils doivent trouver dans les logs :**

```
192.168.10.10 - - [12/Oct/2023:10:15:30 +0000] "GET /confidentiel/flag.txt HTTP/1.1" 200 245 "-" "curl/7.68.0"
```

**Analyse :**

- Qui ? `IP 192.168.10.10` (Pi-Attaquant)
- Quoi ? Accès au fichier `flag.txt`
- Comment ? Outil `curl`
- Résultat ? `Code 200` (Succès, le fichier a été livré)

